# مسح شامل لكل مؤشرات `pandas_ta_classic` — كود اختبار + فرضيات مُولَّدة آلياً

هذا الدفتر **مستقلّ بذاته** (لا يعتمد على أي دفتر آخر في المشروع عبر `%run`) — غرضه مسح **كل** المؤشرات الـ193 المتاحة في `pandas_ta_classic` (تسع فئات: overlap، trend، statistics، cycles، performance، volatility، volume، candles، momentum)، لا مجموعة مُنتقاة يدوياً كما في `signal_discovery_lab.ipynb` (21 مرشّحاً فقط حتى الآن).

**ما يفعله فعلياً:**
1. يبني بيانات OHLCV وهمية (عشوائية، لا حاجة لـDrive) لاختبار أن كل مؤشر يعمل رياضياً بلا أخطاء — **لا لقياس أي قيمة تنبّئية حقيقية** (ذلك يحتاج بيانات حقيقية وسيتم لاحقاً عبر محور `signal_evaluation_axis` إن رغبتَ).
2. يختبر كل مؤشر **بفترات مختلفة** (`length` = 7/14/21/50 للمؤشرات ذات الطول الواحد، وثلاث توليفات fast/slow/signal للمؤشرات المزدوجة كـMACD) — لا فترة افتراضية واحدة فقط.
3. يُصنّف كل نتيجة: نجحت فعلياً (قيم متغيّرة، لا NaN بالكامل)، أم ثابتة (بلا معلومة)، أم فشلت (خطأ صريح، عادة لأنها تحتاج مدخلاً خارجياً غير OHLCV كـ`mavp`/`correl`/`beta`).
4. يُولّد فرضية نصّية عامة لكل فئة (**بلا افتراض اتجاه مسبق** — راجع القسم ٤ لسبب ذلك) وقائمة مرشّحين جاهزة الصيغة لدمجها لاحقاً في `signal_discovery_lab.ipynb`.
5. يحفظ كل النتائج في ملفَي JSON (`pandas_ta_survey_results.json` و`pandas_ta_survey_candidates.json`) في نفس مجلد الدفتر، ويطبع تقريراً نهائياً كاملاً.

**شغّله بـ Runtime → Run all في Colab** — كل خلية تعتمد على سابقتها بالترتيب فقط، لا شيء يحتاج تدخّلاً يدوياً.

## ١) التجهيز — تثبيت واستيراد `pandas_ta_classic`

In [ ]:
# @title
try:
    import pandas_ta_classic as ta
except ImportError:
    %pip install -q pandas_ta_classic
    import pandas_ta_classic as ta

import pandas as pd, numpy as np, inspect, warnings, json
warnings.filterwarnings('ignore')

print('pandas_ta_classic version:', getattr(ta, 'version', '?'))
print('إجمالي المؤشرات المُصنَّفة:', sum(len(v) for v in ta.Category.values()), 'عبر', len(ta.Category), 'فئات')

## ٢) بيانات OHLCV وهمية — عشوائية، بلا أي اعتماد على Drive

In [ ]:
# @title
def make_dummy_ohlcv(n=300, seed=0):
    """مسيرة عشوائية واقعية بما يكفي (اتجاه + تقلّب + حجم) لاختبار أن كل
    مؤشر يُنتج قيماً متغيّرة فعلاً، لا لقياس أي قوة تنبّئية — ذلك يحتاج
    بيانات حقيقية عبر محور signal_evaluation_axis، خطوة لاحقة منفصلة."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range('2024-01-01', periods=n, freq='D')
    close = 100 * np.exp(np.cumsum(rng.normal(0, 0.02, n)))
    openp = np.r_[close[0], close[:-1]]
    high = np.maximum(openp, close) * (1 + rng.random(n) * 0.01)
    low = np.minimum(openp, close) * (1 - rng.random(n) * 0.01)
    volume = rng.random(n) * 1000 + 100
    return pd.DataFrame({'open': openp, 'high': high, 'low': low, 'close': close,
                         'volume': volume}, index=idx)


DUMMY_DF = make_dummy_ohlcv()
print('شكل البيانات الوهمية:', DUMMY_DF.shape)
DUMMY_DF.tail(3)

## ٣) المِسبار — يستدعي كل مؤشر بفترات مختلفة ويتحقّق من صحّة الناتج

**بفترات مختلفة، كما طُلِب صراحةً**: أي مؤشر فيه معامل `length` يُختبَر عند `[7, 14, 21, 50]` (قصير/متوسط/شائع/طويل) بدل فترة افتراضية واحدة؛ أي مؤشر بمعاملَي `fast`/`slow` (وربما `signal`) كـMACD/PPO/KST يُختبَر بثلاث توليفات (`12/26/9` الكلاسيكية، `6/13/5` أقصر، `24/52/18` أطول). المؤشرات بلا معامل طول (كـ`obv`/`vwap`/`ohlc4`) تُختبَر مرّة واحدة بإعداداتها الافتراضية فقط.

**تحذير مُكتشَف أثناء بناء هذا المِسبار (يستحقّ معرفته قبل الوثوق بأي نتيجة `ok`)**: `pandas_ta` تُعوّض أعمدة OHLCV الناقصة ضمنياً بدل رفع خطأ صريح — تحقّقتُ فعلياً أن `atr` (يحتاج مفهومياً `high`/`low`) **ينجح حتى مع عمود `close` وحده**، بنتيجة عديمة المعنى لا خطأ واضح. لذا **نجاح الاستدعاء هنا لا يعني أن المؤشر متوافق مع `feature_order` الحالي لخط الأنابيب** (الذي يستبعد `open`/`high`/`low`/`volume` افتراضياً) — التحقّق من ذلك يحتاج بيانات حقيقية كاملة العواميد لاحقاً، لا افتراضاً من نجاح هذا المسح وحده.

In [ ]:
# @title
LENGTH_SWEEP = [7, 14, 21, 50]
FAST_SLOW_SWEEP = [(12, 26, 9), (6, 13, 5), (24, 52, 18)]


def param_variants(sig_params):
    names = set(sig_params)
    if 'length' in names:
        return [{'length': L} for L in LENGTH_SWEEP]
    if {'fast', 'slow'}.issubset(names):
        variants = []
        for f, s, sg in FAST_SLOW_SWEEP:
            kw = {'fast': f, 'slow': s}
            if 'signal' in names:
                kw['signal'] = sg
            variants.append(kw)
        return variants
    return [{}]


def run_survey(df=None, indicators=None):
    """يستدعي كل مؤشر (بكل توليفة فترات) على df، ويُصنِّف الناتج:
    'ok' (قيم صالحة متغيّرة)، 'constant' (نجح لكن بلا معلومة فعلية)،
    'all_nan'، أو 'error' (استثناء صريح — عادة معامل خارجي إلزامي مفقود)."""
    df = df if df is not None else DUMMY_DF
    names = (sorted({nm for cat in ta.Category.values() for nm in cat})
             if indicators is None else indicators)
    cat_of = {nm: cat for cat, lst in ta.Category.items() for nm in lst}
    rows = []
    for name in names:
        fn = getattr(df.ta, name, None)
        if fn is None:
            rows.append(dict(indicator=name, category=cat_of.get(name, '?'), params={},
                             status='no_attr', n_cols=0, n_valid=0, is_constant=None,
                             error='', columns=[]))
            continue
        try:
            sig = inspect.signature(fn)
        except (TypeError, ValueError) as e:
            rows.append(dict(indicator=name, category=cat_of.get(name, '?'), params={},
                             status='no_signature', n_cols=0, n_valid=0, is_constant=None,
                             error=str(e), columns=[]))
            continue
        for kw in param_variants(sig.parameters):
            try:
                out = fn(**kw)
            except Exception as e:
                rows.append(dict(indicator=name, category=cat_of.get(name, '?'), params=kw,
                                 status='error', n_cols=0, n_valid=0, is_constant=None,
                                 error=f'{type(e).__name__}: {e}', columns=[]))
                continue
            if out is None:
                rows.append(dict(indicator=name, category=cat_of.get(name, '?'), params=kw,
                                 status='returned_none', n_cols=0, n_valid=0, is_constant=None,
                                 error='', columns=[]))
                continue
            out_df = out.to_frame() if isinstance(out, pd.Series) else out
            n_valid = int((~out_df.isna().all(axis=1)).sum())
            col_nunique = out_df.apply(lambda c: c.dropna().nunique())
            is_constant = bool((col_nunique <= 1).all()) if n_valid > 0 else None
            status = ('ok' if (n_valid > 0 and not is_constant)
                     else ('all_nan' if n_valid == 0 else 'constant'))
            rows.append(dict(indicator=name, category=cat_of.get(name, '?'), params=kw,
                             status=status, n_cols=out_df.shape[1], n_valid=n_valid,
                             is_constant=is_constant, error='',
                             columns=list(out_df.columns)))
    return pd.DataFrame(rows)


print('✅ المِسبار جاهز — لم يُشغَّل بعد (الخلية التالية تُشغّله فعلياً).')

## ٤) التشغيل الفعلي — كل الـ193 مؤشراً على البيانات الوهمية

In [ ]:
# @title
SURVEY_RESULTS = run_survey()

print('إجمالي التوليفات (مؤشر × فترة):', len(SURVEY_RESULTS))
print()
print('التوزيع حسب الحالة:')
print(SURVEY_RESULTS['status'].value_counts().to_string())
print()
print('عدد المؤشرات الناجحة (ok) — فريدة لا حسب الفترة — لكل فئة:')
print(SURVEY_RESULTS[SURVEY_RESULTS.status == 'ok'].groupby('category').indicator.nunique().to_string())

### تفاصيل الحالات غير الناجحة (للمراجعة، لا للقلق — أغلبها متوقَّع)

In [ ]:
# @title
problems = SURVEY_RESULTS[SURVEY_RESULTS.status.isin(['error', 'constant', 'all_nan', 'no_attr', 'no_signature', 'returned_none'])]
if len(problems):
    print(problems[['indicator', 'category', 'params', 'status', 'error']].to_string(index=False))
else:
    print('لا مشاكل — كل المؤشرات نجحت في كل توليفة فترات.')

## ٥) فرضيات عامة لكل فئة — بلا افتراض اتجاه مسبق

**لماذا بلا اتجاه مسبق (لا "ارتداد" ولا "استمرار" مُقرَّراً سلفاً)؟** محور `signal_evaluation_axis` يقيس ارتباط سبيرمان (IC) بين المرشّح والعائد الفعلي — وهذا **متماثل تحت انعكاس الإشارة**: لو كانت `X` ذات IC سالب ثابت عبر كل النوافذ، فـ`-X` ستكون بنفس القوة بالضبط IC موجباً ثابتاً؛ كلاهما "مقبول" بنفس معيار `consistent_sign` + `frac_significant`. الاتجاه المُفترَض مسبقاً (كما في `RSI_14_reversion` القديمة) تسمية تفسيرية للبشر فقط، **لا شرطاً إحصائياً** — وقد أثبتت H001 أن الاتجاه "البديهي" (الزخم) كان معكوساً فعلياً (انعكاس، لا استمرار). لذا هذا المسح يُسجّل القيمة **الخام** فقط لكل مؤشر، ويترك للمحور نفسه اكتشاف الإشارة (أيّاً كانت) لاحقاً.

In [ ]:
# @title
CATEGORY_HYPOTHESES = {
    'momentum': 'زخم/تذبذب في السعر أو معدّل تغيّره — قد يدلّ على استمرار الاتجاه أو ارتداد عن تطرّف، حسب الإشارة الفعلية لـmean_ic لا افتراض مسبق.',
    'overlap': 'مستوى سعري مُشتقّ (متوسط متحرّك، نطاق، بيفوت) — **مقياسه بوحدة السعر لا نسبة مئوية**؛ يُنصَح بتحويله لمسافة نسبية عن close (مثال: (close/value-1)*100) قبل تجميعه عبر أصول مختلفة الأسعار، بنفس منطق استبعاد high/low/close الخام في crypto_data_pipeline_v6.ipynb.',
    'trend': 'قياس قوة/اتجاه/تحوّل نظام سعري — لا سعرياً بحتاً غالباً، لكن راجع كل مؤشر (بعضها كـpsar/supertrend بوحدة سعرية).',
    'statistics': 'خاصية إحصائية بحتة للسلسلة (انحراف، التواء، تفرطح، اعتدال) — تفسيرها السلوكي غير مباشر، يحتاج فرضية أدقّ لكل حالة.',
    'cycles': 'محاولة كشف دورة زمنية متكرّرة (Hilbert Transform) — تجريبية بطبيعتها، أضعف تفسيراً سلوكياً من البقية.',
    'performance': 'عائد/تراجع تراكمي — أقرب لتوصيف الأداء الماضي منه لإشارة تنبّؤية مباشرة؛ أنسب كأساس مقارنة (baseline) لا كمرشّح أساسي.',
    'volatility': 'مقياس تقلّب (نطاق، ATR، Bollinger) — الفرضية الأقرب فعلياً لما أثبتته H003: تقلّب مرتفع قد يسبق انكماشاً/ارتداداً، لا استمرار تقلّب بالضرورة.',
    'volume': 'مقياس مبنيّ على الحجم (تدفّق نقدي، عدم توازن شراء/بيع) — يحتاج حجماً حقيقياً ذا دلالة اتجاهية؛ راجع حالة vfi أدناه (ثابتة على بيانات وهمية عشوائية الحجم تماماً — قد تحتاج حجماً واقعياً مترابطاً بالسعر لتُنتج قيماً فعلياً).',
    'candles': 'نمط شمعة/مجموعة شموع (دوجي، إنجلافينغ...) — مخرجه غالباً منفصل (-100/0/100)، يحتاج تعاملاً كفئة لا كقيمة مستمرة.',
}

for cat, txt in CATEGORY_HYPOTHESES.items():
    print(f'[{cat}] {txt}\n')

## ٦) توليد قائمة مرشّحين جاهزة — بصيغة قابلة للدمج في `signal_discovery_lab.ipynb`

لكل توليفة (مؤشر، فترة) نجحت فعلياً (`status == 'ok'`)، يُبنى قاموس مرشّح بنفس بنية `CANDIDATE_SIGNALS`/`EXPLORATORY_CANDIDATES` هناك (`name`، `track`، `hypothesis`) — **العمود الأول** من ناتج كل مؤشر يُستخدَم كقيمة تمثيلية (المؤشرات متعدّدة الأعمدة كـMACD/BBANDS تظهر كل أعمدتها في عمود `columns` بجدول `SURVEY_RESULTS` لمن يريد تجربة عمود آخر لاحقاً).

In [ ]:
# @title
def build_candidate_dicts(results_df):
    ok = results_df[results_df.status == 'ok'].copy()
    candidates = []
    for _, row in ok.iterrows():
        param_suffix = '_'.join(f'{k}{v}' for k, v in row['params'].items()) or 'default'
        candidates.append({
            'name': f"{row['indicator'].upper()}_{param_suffix}",
            'track': 'literature_mining',
            'source_indicator': row['indicator'],
            'category': row['category'],
            'params': row['params'],
            'primary_column': row['columns'][0] if row['columns'] else None,
            'all_columns': row['columns'],
            'hypothesis': CATEGORY_HYPOTHESES.get(row['category'], 'بلا فرضية فئة محدَّدة.'),
        })
    return candidates


SURVEY_CANDIDATES = build_candidate_dicts(SURVEY_RESULTS)
print('عدد المرشّحين المُولَّدين آلياً (ok فقط):', len(SURVEY_CANDIDATES))
print('مثال على أول 3 مرشّحين:')
for c in SURVEY_CANDIDATES[:3]:
    print(c)

## ٧) الحفظ — نتائج المسح الكامل + قائمة المرشّحين

In [ ]:
# @title
results_export = SURVEY_RESULTS.copy()
results_export['params'] = results_export['params'].apply(json.dumps)
results_export['columns'] = results_export['columns'].apply(json.dumps)
results_export.to_json('pandas_ta_survey_results.json', orient='records', force_ascii=False, indent=2)

with open('pandas_ta_survey_candidates.json', 'w', encoding='utf-8') as f:
    json.dump(SURVEY_CANDIDATES, f, ensure_ascii=False, indent=2)

print('✅ حُفظ: pandas_ta_survey_results.json (', len(results_export), 'صفّاً)')
print('✅ حُفظ: pandas_ta_survey_candidates.json (', len(SURVEY_CANDIDATES), 'مرشّحاً)')

## ٨) التقرير النهائي — الخطوة التالية

In [ ]:
# @title
n_total_indicators = sum(len(v) for v in ta.Category.values())
n_ok_indicators = SURVEY_RESULTS[SURVEY_RESULTS.status == 'ok'].indicator.nunique()
n_problem_indicators = SURVEY_RESULTS[~SURVEY_RESULTS.status.isin(['ok'])].indicator.nunique()
hard_errors = SURVEY_RESULTS[SURVEY_RESULTS.status == 'error']['indicator'].unique().tolist()
_ok_set = set(SURVEY_RESULTS[SURVEY_RESULTS.status == 'ok'].indicator)
always_constant = sorted(set(SURVEY_RESULTS[SURVEY_RESULTS.status == 'constant'].indicator) - _ok_set)

print('=' * 70)
print('تقرير المسح الشامل لـ pandas_ta_classic')
print('=' * 70)
print(f'إجمالي المؤشرات المُصنَّفة في المكتبة : {n_total_indicators}')
print(f'نجحت (ok) في فترة واحدة على الأقل      : {n_ok_indicators}')
print(f'مرشّحون جاهزون مُولَّدون آلياً          : {len(SURVEY_CANDIDATES)}')
print(f'مؤشرات بها مشاكل في كل فتراتها المُختبَرة: {n_problem_indicators}')
print(f'  منها فشل صريح (يحتاج مدخلاً خارجياً)  : {hard_errors}')
print(f'  منها ثابتة في كل الفترات المُختبَرة    : {always_constant}')
print()
print('⚠️ تذكير مهم (راجع القسم ٣): نجاح الاستدعاء هنا (ok) لا يعني توافقاً')
print('مضموناً مع feature_order الحالي لخط الأنابيب — pandas_ta تُعوّض أعمدة')
print('OHLCV الناقصة ضمنياً بدل رفع خطأ، فقد ينجح مؤشر هنا بلا معنى فعلي إن')
print('غابت بياناته الحقيقية لاحقاً. التحقّق النهائي يحتاج تشغيلاً فعلياً على')
print('dataset حقيقي كامل الأعمدة (كما فعلت crypto_data_pipeline_v6.ipynb).')
print()
print('الخطوة التالية (قرار صريح مطلوب، لا تنفيذ تلقائي):')
print('  1. اختيار عيّنة من الـ', len(SURVEY_CANDIDATES), 'مرشّحاً (أو كلّها دفعة واحدة عبر')
print('     run_batch_and_register) لتقييمها فعلياً عبر signal_evaluation_axis على')
print('     بيانات حقيقية — يحتاج أولاً توفّر الأعمدة الخام التي يحتاجها كل مؤشر في')
print('     feature_order (راجع سابقة FRACTAL_high/low في crypto_data_pipeline_v6.ipynb')
print('     كمثال على كيفية إضافة ميزة سببية جديدة بأمان دون كسر التوافق الخلفي).')
print('  2. مرشّحو فئة overlap تحديداً يحتاجون تحويلاً لمسافة نسبية عن close قبل')
print('     أي اختبار (راجع القسم ٥) — لا اختبار قيمتها الخام كما هي.')
print('  3. vfi ثابتة على كل الفترات المُختبَرة هنا (حجم عشوائي بلا دلالة اتجاهية) —')
print('     يستحقّ إعادة تجربتها على بيانات حقيقية قبل استبعادها نهائياً.')
print('=' * 70)